# Phase 1, test P1b: were the gray-lift draws identical to single-threaded draws? (CPU only)

`gray-lift-kill-test` generated the gray noise with NumPy's *global* RNG (`np.random.seed(seed)` then
`np.random.normal`) inside two GPU worker threads, so a draw can interleave with the other thread's. The Xformer,
GRL-B, SCUNet and DRUNet reruns drew single-threaded. Restormer-gray ran inside the lift job on the lift's own draws,
so rerunning it here on single-threaded draws reproduces its per-image PSNR iff the draws were identical.

Pre-registered (decision log, Phase 1): same draw ⇔ |Δ PSNR| ≤ 0.003 dB (CPU vs GPU numerics); Set12 + BSD68,
σ 15 / 25 / 50, for the seeds in REF. Any mismatch voids the identical-draw pairing for that image and seed.
Kaggle: CPU, internet on.

In [ ]:
import importlib.util, json, math, os, time, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

torch.set_num_threads(os.cpu_count())
SIGMAS = [15, 25, 50]
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
REF = json.loads(r'''{"Set12|01.png|15|1": 32.96543695906435, "Set12|02.png|15|1": 36.27062909513738, "Set12|03.png|15|1": 33.74644295425924, "Set12|04.png|15|1": 32.729835932924814, "Set12|05.png|15|1": 33.69453350083947, "Set12|06.png|15|1": 32.0953998158703, "Set12|07.png|15|1": 32.1751256196039, "Set12|08.png|15|1": 35.023640254330765, "Set12|09.png|15|1": 33.831654958419044, "Set12|10.png|15|1": 32.956798454899406, "Set12|11.png|15|1": 32.67460537692757, "BSD68|test001.png|15|1": 28.508524750361374, "Set12|12.png|15|1": 32.921795597395025, "BSD68|test002.png|15|1": 32.97620895145307, "BSD68|test003.png|15|1": 33.01114571917728, "BSD68|test005.png|15|1": 30.31935460696168, "BSD68|test004.png|15|1": 33.55102889478302, "BSD68|test006.png|15|1": 35.49205900747013, "BSD68|test007.png|15|1": 31.5712464001574, "BSD68|test008.png|15|1": 29.2038132527766, "BSD68|test009.png|15|1": 31.796948572146576, "BSD68|test010.png|15|1": 32.3740837605146, "BSD68|test011.png|15|1": 32.44356611697241, "BSD68|test012.png|15|1": 31.22524280736939, "BSD68|test013.png|15|1": 32.2132832352071, "BSD68|test014.png|15|1": 33.7054502030653, "BSD68|test015.png|15|1": 30.09443265246048, "BSD68|test016.png|15|1": 31.48184287545851, "BSD68|test017.png|15|1": 35.27786964015955, "BSD68|test018.png|15|1": 35.76448845922988, "BSD68|test019.png|15|1": 31.00969049533549, "BSD68|test020.png|15|1": 31.785215280401395, "BSD68|test021.png|15|1": 29.66832417087013, "BSD68|test022.png|15|1": 30.18756971466896, "BSD68|test023.png|15|1": 32.215636381629835, "BSD68|test024.png|15|1": 31.637628836555685, "BSD68|test025.png|15|1": 32.390066036418, "BSD68|test026.png|15|1": 31.233821393443517, "BSD68|test027.png|15|1": 32.74100736163609, "BSD68|test028.png|15|1": 35.72762403245568, "BSD68|test029.png|15|1": 27.724812690284665, "BSD68|test030.png|15|1": 32.59531994182446, "BSD68|test031.png|15|1": 27.83688723270709, "BSD68|test032.png|15|1": 29.570651426141357, "BSD68|test033.png|15|1": 30.908501518520804, "BSD68|test034.png|15|1": 35.72017746147482, "BSD68|test035.png|15|1": 30.766988000754377, "BSD68|test036.png|15|1": 29.39885832366269, "BSD68|test037.png|15|1": 30.94916244118988, "BSD68|test038.png|15|1": 31.987051999880677, "BSD68|test039.png|15|1": 37.33631749600477, "BSD68|test040.png|15|1": 32.03803606698202, "BSD68|test041.png|15|1": 30.796864595596595, "BSD68|test042.png|15|1": 32.51733269379034, "BSD68|test044.png|15|1": 31.41135376947361, "BSD68|test043.png|15|1": 33.44250860346288, "BSD68|test045.png|15|1": 36.5024318586785, "BSD68|test046.png|15|1": 29.3426818072906, "BSD68|test047.png|15|1": 29.42957489896505, "BSD68|test048.png|15|1": 33.469564377563, "BSD68|test049.png|15|1": 33.557431009711244, "BSD68|test050.png|15|1": 29.5676324937449, "BSD68|test051.png|15|1": 31.454723099050675, "BSD68|test052.png|15|1": 34.264599318343976, "BSD68|test053.png|15|1": 33.39499681312718, "BSD68|test054.png|15|1": 32.06774014074392, "BSD68|test055.png|15|1": 30.436766204330503, "BSD68|test056.png|15|1": 27.07588387785925, "BSD68|test057.png|15|1": 30.950909357621303, "BSD68|test058.png|15|1": 32.813217460372755, "BSD68|test059.png|15|1": 32.49185211865098, "BSD68|test060.png|15|1": 33.32076606897273, "BSD68|test061.png|15|1": 32.81343572855314, "BSD68|test062.png|15|1": 37.21023969442827, "BSD68|test063.png|15|1": 29.56827786046519, "BSD68|test064.png|15|1": 29.12777364561564, "BSD68|test065.png|15|1": 31.586471344983863, "BSD68|test066.png|15|1": 41.36030267977305, "BSD68|test067.png|15|1": 27.722276539468247, "Set12|01.png|25|1": 30.641726201962094, "BSD68|test068.png|15|1": 29.23381150968384, "Set12|02.png|25|1": 34.21293924329375, "Set12|03.png|25|1": 31.434748347306297, "Set12|04.png|25|1": 30.277076166799752, "Set12|05.png|25|1": 30.9300816138575, "Set12|06.png|25|1": 29.54648103493528, "Set12|07.png|25|1": 29.76766156505881, "Set12|08.png|25|1": 33.012367625565965, "Set12|09.png|25|1": 31.52427347169464, "Set12|10.png|25|1": 30.78974404382126, "Set12|11.png|25|1": 30.353985973804235, "Set12|12.png|25|1": 30.68581547726705, "BSD68|test001.png|25|1": 26.023400663107456, "BSD68|test002.png|25|1": 30.503711395167173, "BSD68|test003.png|25|1": 30.59472930280713, "BSD68|test004.png|25|1": 30.96830104603977, "BSD68|test005.png|25|1": 27.75472257901093, "BSD68|test006.png|25|1": 33.3732093202949, "BSD68|test007.png|25|1": 28.98317755492021, "BSD68|test008.png|25|1": 26.441662072374456, "BSD68|test009.png|25|1": 29.33088467129472, "BSD68|test010.png|25|1": 29.9230757867094, "BSD68|test011.png|25|1": 29.827155541156067, "BSD68|test012.png|25|1": 28.67397369317533, "BSD68|test013.png|25|1": 29.77329082957712, "BSD68|test014.png|25|1": 31.270480762544715, "BSD68|test015.png|25|1": 27.455394386342125, "BSD68|test016.png|25|1": 28.74808048432405, "BSD68|test017.png|25|1": 33.3118349878818, "BSD68|test018.png|25|1": 33.58038996858345, "BSD68|test019.png|25|1": 28.530110830430083, "BSD68|test020.png|25|1": 29.16697868286921, "BSD68|test021.png|25|1": 26.73249664957341, "BSD68|test022.png|25|1": 27.60411007084434, "BSD68|test023.png|25|1": 29.345510721528637, "BSD68|test024.png|25|1": 28.957998529535843, "BSD68|test025.png|25|1": 29.72241094123583, "BSD68|test026.png|25|1": 28.859903545533943, "BSD68|test027.png|25|1": 30.36302175899335, "BSD68|test028.png|25|1": 33.35322667852109, "BSD68|test029.png|25|1": 24.63577888952848, "BSD68|test030.png|25|1": 30.13087827736515, "BSD68|test031.png|25|1": 24.917028997028588, "BSD68|test032.png|25|1": 26.851837344028034, "BSD68|test033.png|25|1": 28.18878518571875, "BSD68|test034.png|25|1": 34.03664756133664, "BSD68|test035.png|25|1": 28.377898835536577, "BSD68|test036.png|25|1": 27.79062111630587, "BSD68|test038.png|25|1": 29.468500271674337, "BSD68|test037.png|25|1": 28.80151356736093, "BSD68|test039.png|25|1": 34.77330842040922, "BSD68|test040.png|25|1": 29.8631370634832, "BSD68|test041.png|25|1": 28.32152278621097, "BSD68|test042.png|25|1": 30.16612242927419, "BSD68|test043.png|25|1": 30.767867562250824, "BSD68|test044.png|25|1": 28.79624670294584, "BSD68|test045.png|25|1": 34.46017078552734, "BSD68|test046.png|25|1": 26.69279445672388, "BSD68|test047.png|25|1": 26.78333456897338, "BSD68|test048.png|25|1": 30.683840719613585, "BSD68|test049.png|25|1": 31.693921148780888, "BSD68|test050.png|25|1": 27.11190055105297, "BSD68|test051.png|25|1": 29.04220329061627, "BSD68|test052.png|25|1": 32.47629247442771, "BSD68|test053.png|25|1": 31.529506212847508, "BSD68|test054.png|25|1": 29.59445437085662, "BSD68|test055.png|25|1": 27.829139192907355, "BSD68|test056.png|25|1": 24.185527305545957, "BSD68|test057.png|25|1": 28.53655210514342, "BSD68|test058.png|25|1": 31.137556985911047, "BSD68|test059.png|25|1": 30.61243071465292, "BSD68|test060.png|25|1": 31.32827764549691, "BSD68|test061.png|25|1": 30.52088596453452, "BSD68|test062.png|25|1": 34.978724012944134, "BSD68|test063.png|25|1": 26.766541050359542, "BSD68|test064.png|25|1": 26.85724563851056, "BSD68|test065.png|25|1": 29.136555410059056, "BSD68|test066.png|25|1": 39.07347396761075, "BSD68|test067.png|25|1": 24.87576549583418, "Set12|01.png|50|1": 27.666940374794123, "Set12|02.png|50|1": 31.462764648482604, "BSD68|test068.png|25|1": 26.2941925735997, "Set12|03.png|50|1": 28.132058907493875, "Set12|04.png|50|1": 26.822636439850854, "Set12|05.png|50|1": 27.51935067007799, "Set12|06.png|50|1": 26.27016622563059, "Set12|07.png|50|1": 26.89593496962423, "Set12|08.png|50|1": 30.110283307273, "Set12|09.png|50|1": 28.32396280241237, "Set12|10.png|50|1": 27.81340359165888, "Set12|11.png|50|1": 27.54339237130305, "BSD68|test001.png|50|1": 23.247401417001235, "Set12|12.png|50|1": 27.68392796378081, "BSD68|test002.png|50|1": 27.63687153040486, "BSD68|test003.png|50|1": 27.48722066677477, "BSD68|test004.png|50|1": 27.98203867004283, "BSD68|test005.png|50|1": 24.946595861446625, "BSD68|test006.png|50|1": 30.72955338736226, "BSD68|test007.png|50|1": 25.85686653358872, "BSD68|test008.png|50|1": 23.2803460721699, "BSD68|test009.png|50|1": 26.397916932763064, "BSD68|test010.png|50|1": 27.027475171432243, "BSD68|test011.png|50|1": 26.5345287257877, "BSD68|test012.png|50|1": 25.57109023089598, "BSD68|test013.png|50|1": 27.007199742426543, "BSD68|test014.png|50|1": 28.28645878850821, "BSD68|test015.png|50|1": 24.44193553871868, "BSD68|test016.png|50|1": 25.05022202201307, "BSD68|test017.png|50|1": 30.95637590270264, "BSD68|test018.png|50|1": 31.016230119860904, "BSD68|test019.png|50|1": 25.858220407132336, "BSD68|test020.png|50|1": 26.06981906441717, "BSD68|test021.png|50|1": 23.264339305873172, "BSD68|test022.png|50|1": 24.428630507688727, "BSD68|test023.png|50|1": 25.82276612920453, "BSD68|test024.png|50|1": 25.623684342931003, "BSD68|test026.png|50|1": 25.971996336842587, "BSD68|test025.png|50|1": 26.558138281367228, "BSD68|test027.png|50|1": 27.405961472460888, "BSD68|test028.png|50|1": 30.630568673759587, "BSD68|test029.png|50|1": 21.153190640418902, "BSD68|test030.png|50|1": 27.194855887526856, "BSD68|test031.png|50|1": 21.592232513024975, "BSD68|test032.png|50|1": 23.673358643744173, "BSD68|test033.png|50|1": 25.104551669777678, "BSD68|test034.png|50|1": 31.538751176006382, "BSD68|test035.png|50|1": 25.69178574423432, "BSD68|test036.png|50|1": 26.517306517216078, "BSD68|test037.png|50|1": 26.489189890322955, "BSD68|test038.png|50|1": 26.42933280973621, "BSD68|test039.png|50|1": 30.904241072201124, "BSD68|test040.png|50|1": 26.990465333426386, "BSD68|test041.png|50|1": 25.446401486566263, "BSD68|test042.png|50|1": 27.10046381662416, "BSD68|test043.png|50|1": 27.32117579151804, "BSD68|test044.png|50|1": 25.627667669846925, "BSD68|test045.png|50|1": 31.89956859681032, "BSD68|test046.png|50|1": 23.49158945471408, "BSD68|test047.png|50|1": 23.84984601023016, "BSD68|test048.png|50|1": 26.887705647537004, "BSD68|test049.png|50|1": 29.602240142373827, "BSD68|test050.png|50|1": 24.417924999691355, "BSD68|test051.png|50|1": 25.86776765593544, "BSD68|test052.png|50|1": 30.147893935519377, "BSD68|test053.png|50|1": 29.47969951056062, "BSD68|test054.png|50|1": 26.43779108341033, "BSD68|test055.png|50|1": 24.722962138337216, "BSD68|test056.png|50|1": 21.23181893478085, "BSD68|test057.png|50|1": 25.986507878048517, "BSD68|test058.png|50|1": 29.37333398236925, "BSD68|test059.png|50|1": 28.64318181905994, "BSD68|test060.png|50|1": 29.13214778526816, "BSD68|test061.png|50|1": 27.86452464754857, "BSD68|test062.png|50|1": 31.67768338089583, "BSD68|test063.png|50|1": 23.55313793210068, "BSD68|test064.png|50|1": 24.38277638044496, "BSD68|test065.png|50|1": 26.372148008485745, "BSD68|test067.png|50|1": 21.61515595536164, "BSD68|test066.png|50|1": 35.714433198141904, "Set12|01.png|15|2": 32.973075069028695, "Set12|02.png|15|2": 36.13905517278922, "BSD68|test068.png|50|1": 23.070018696508903, "Set12|03.png|15|2": 33.76441427500243, "Set12|04.png|15|2": 32.7007466139264, "Set12|05.png|15|2": 33.642204488139186, "Set12|06.png|15|2": 32.10748920936553, "Set12|07.png|15|2": 32.18582232815099, "Set12|08.png|15|2": 35.02556624212933, "Set12|09.png|15|2": 33.73424519552752, "Set12|10.png|15|2": 32.95707780628163, "Set12|11.png|15|2": 32.64701084587692, "BSD68|test001.png|15|2": 28.47730924655083, "Set12|12.png|15|2": 32.90061746198675, "BSD68|test002.png|15|2": 32.99349224975168, "BSD68|test003.png|15|2": 32.974963634553006, "BSD68|test004.png|15|2": 33.61606851857979, "BSD68|test005.png|15|2": 30.324125641975236, "BSD68|test007.png|15|2": 31.56412298128654, "BSD68|test006.png|15|2": 35.47275583018914, "BSD68|test008.png|15|2": 29.15437675006741, "BSD68|test009.png|15|2": 31.761352112145367, "BSD68|test010.png|15|2": 32.35747593496033, "BSD68|test011.png|15|2": 32.45061353232209, "BSD68|test012.png|15|2": 31.19765524466905, "BSD68|test013.png|15|2": 32.22301163440431, "BSD68|test014.png|15|2": 33.62576502297692, "BSD68|test015.png|15|2": 30.06456591618297, "BSD68|test016.png|15|2": 31.447346215648505, "BSD68|test017.png|15|2": 35.2723171457872, "BSD68|test018.png|15|2": 35.74841845126243, "BSD68|test019.png|15|2": 30.959044864312546, "BSD68|test020.png|15|2": 31.75703053003428, "BSD68|test021.png|15|2": 29.670880981763258, "BSD68|test022.png|15|2": 30.181277694568745, "BSD68|test023.png|15|2": 32.2448320242288, "BSD68|test024.png|15|2": 31.5813153070491, "BSD68|test025.png|15|2": 32.39318264856345, "BSD68|test026.png|15|2": 31.170696049310887, "BSD68|test027.png|15|2": 32.72403740994629, "BSD68|test028.png|15|2": 35.734151892264514, "BSD68|test029.png|15|2": 27.69624406076865, "BSD68|test030.png|15|2": 32.6168374559512, "BSD68|test031.png|15|2": 27.79620119037751, "BSD68|test032.png|15|2": 29.546445906890163, "BSD68|test033.png|15|2": 30.940816555663027, "BSD68|test034.png|15|2": 35.71691391687922, "BSD68|test035.png|15|2": 30.769106167096187, "BSD68|test036.png|15|2": 29.390098717185364, "BSD68|test037.png|15|2": 30.971334926694407, "BSD68|test038.png|15|2": 31.93182323342977, "BSD68|test039.png|15|2": 37.31459572828774, "BSD68|test040.png|15|2": 31.998287386806577, "BSD68|test041.png|15|2": 30.75941998740952, "BSD68|test042.png|15|2": 32.43457550757811, "BSD68|test043.png|15|2": 33.48190814095962, "BSD68|test044.png|15|2": 31.39541049084339, "BSD68|test046.png|15|2": 29.31674122815801, "BSD68|test045.png|15|2": 36.46952465113765, "BSD68|test047.png|15|2": 29.461422290347933, "BSD68|test048.png|15|2": 33.43401915229984, "BSD68|test049.png|15|2": 33.57243726385518, "BSD68|test050.png|15|2": 29.518680754933865, "BSD68|test051.png|15|2": 31.464097901087133, "BSD68|test052.png|15|2": 34.296922578459665, "BSD68|test053.png|15|2": 33.37622383910626, "BSD68|test054.png|15|2": 32.10525744455208, "BSD68|test055.png|15|2": 30.419205315479743, "BSD68|test056.png|15|2": 27.08046280422247, "BSD68|test057.png|15|2": 30.895279754114853, "BSD68|test058.png|15|2": 32.83736030586887, "BSD68|test059.png|15|2": 32.50289074577509, "BSD68|test060.png|15|2": 33.25849430387195, "BSD68|test061.png|15|2": 32.78342779459488, "BSD68|test062.png|15|2": 37.31526473235945, "BSD68|test063.png|15|2": 29.54178443354941, "BSD68|test064.png|15|2": 29.115447463520027, "BSD68|test065.png|15|2": 31.53025803178212, "BSD68|test066.png|15|2": 41.271599466467386, "BSD68|test067.png|15|2": 27.66994550115253, "Set12|01.png|25|2": 30.64909256218688, "BSD68|test068.png|15|2": 29.24247991150348, "Set12|02.png|25|2": 34.22334563117122, "Set12|03.png|25|2": 31.45690117784182, "Set12|04.png|25|2": 30.1893876955624, "Set12|05.png|25|2": 30.813427823370358, "Set12|06.png|25|2": 29.533447602689503, "Set12|07.png|25|2": 29.738426713085403, "Set12|08.png|25|2": 33.003885685774, "Set12|09.png|25|2": 31.42395215505099, "Set12|10.png|25|2": 30.757108693800745, "Set12|11.png|25|2": 30.32889646901362, "Set12|12.png|25|2": 30.62889362467603, "BSD68|test001.png|25|2": 26.0201352849883, "BSD68|test002.png|25|2": 30.55678588554793, "BSD68|test003.png|25|2": 30.61198595423075, "BSD68|test004.png|25|2": 31.055180164743664, "BSD68|test005.png|25|2": 27.728026301377405, "BSD68|test006.png|25|2": 33.349796798067985, "BSD68|test007.png|25|2": 28.99860511871577, "BSD68|test008.png|25|2": 26.410602567382877, "BSD68|test009.png|25|2": 29.26682978544465, "BSD68|test010.png|25|2": 29.900585051944176, "BSD68|test011.png|25|2": 29.830701991618483, "BSD68|test012.png|25|2": 28.64201472843594, "BSD68|test013.png|25|2": 29.78187068437223, "BSD68|test014.png|25|2": 31.26601230012149, "BSD68|test015.png|25|2": 27.43497958868597, "BSD68|test016.png|25|2": 28.70534148676912, "BSD68|test017.png|25|2": 33.290338830325375, "BSD68|test018.png|25|2": 33.59936209782045, "BSD68|test019.png|25|2": 28.47286045304197, "BSD68|test020.png|25|2": 29.152360997197725, "BSD68|test021.png|25|2": 26.690454206839597, "BSD68|test022.png|25|2": 27.584535750518697, "BSD68|test023.png|25|2": 29.428944506764868, "BSD68|test024.png|25|2": 28.908367586344976, "BSD68|test025.png|25|2": 29.694446880837138, "BSD68|test026.png|25|2": 28.839745596575064, "BSD68|test027.png|25|2": 30.34095783584975, "BSD68|test028.png|25|2": 33.390525177306635, "BSD68|test029.png|25|2": 24.607902082670616, "BSD68|test030.png|25|2": 30.154077682491483, "BSD68|test031.png|25|2": 24.890119687799416, "BSD68|test032.png|25|2": 26.82577504899971, "BSD68|test033.png|25|2": 28.19439702971112, "BSD68|test034.png|25|2": 34.004978368764085, "BSD68|test035.png|25|2": 28.3738754920575, "BSD68|test036.png|25|2": 27.782097644807934, "BSD68|test038.png|25|2": 29.441261296940645, "BSD68|test037.png|25|2": 28.807635481910573, "BSD68|test039.png|25|2": 34.71292807402538, "BSD68|test040.png|25|2": 29.800884168321925, "BSD68|test041.png|25|2": 28.30483232526472, "BSD68|test042.png|25|2": 30.07239589934641, "BSD68|test043.png|25|2": 30.80652692865165, "BSD68|test044.png|25|2": 28.75622060786626, "BSD68|test045.png|25|2": 34.49355166525556, "BSD68|test046.png|25|2": 26.665713419369773, "BSD68|test047.png|25|2": 26.805391661588025, "BSD68|test048.png|25|2": 30.6036077728624, "BSD68|test049.png|25|2": 31.740538739782377, "BSD68|test050.png|25|2": 27.0640417850931, "BSD68|test051.png|25|2": 29.04882895301475, "BSD68|test052.png|25|2": 32.49282654584601, "BSD68|test053.png|25|2": 31.52605210280122, "BSD68|test054.png|25|2": 29.634097681565503, "BSD68|test055.png|25|2": 27.83477873114169, "BSD68|test056.png|25|2": 24.19245772678598, "BSD68|test057.png|25|2": 28.467172663490445, "BSD68|test058.png|25|2": 31.152459487253328, "BSD68|test059.png|25|2": 30.62778716337283, "BSD68|test060.png|25|2": 31.29749582347856, "BSD68|test061.png|25|2": 30.52840921853805, "BSD68|test062.png|25|2": 35.026702678046355, "BSD68|test063.png|25|2": 26.72343656290636, "BSD68|test064.png|25|2": 26.84928879114201, "BSD68|test065.png|25|2": 29.106263847117, "BSD68|test066.png|25|2": 38.99896476784864, "BSD68|test067.png|25|2": 24.828789772825893, "Set12|01.png|50|2": 27.57637492912808, "Set12|02.png|50|2": 31.60770176960655, "BSD68|test068.png|25|2": 26.299667969724403, "Set12|03.png|50|2": 28.048774594756885, "Set12|04.png|50|2": 26.72905521834596, "Set12|05.png|50|2": 27.16461547881721, "Set12|06.png|50|2": 26.22536357471184, "Set12|07.png|50|2": 26.851588597482767, "Set12|08.png|50|2": 30.229127875761243, "Set12|09.png|50|2": 28.33741948342936, "Set12|10.png|50|2": 27.78592221876885, "Set12|11.png|50|2": 27.53660795095993, "BSD68|test001.png|50|2": 23.25683251000341, "Set12|12.png|50|2": 27.63151647561676, "BSD68|test002.png|50|2": 27.663509203693323, "BSD68|test003.png|50|2": 27.535670877147137, "BSD68|test004.png|50|2": 28.03342151192327, "BSD68|test005.png|50|2": 24.8904409936811, "BSD68|test006.png|50|2": 30.82467658268453, "BSD68|test007.png|50|2": 25.808339163636496, "BSD68|test008.png|50|2": 23.258252482262236, "BSD68|test009.png|50|2": 26.292459417889447, "BSD68|test010.png|50|2": 27.019781350013258, "BSD68|test011.png|50|2": 26.537793598223345, "BSD68|test012.png|50|2": 25.504408871599026, "BSD68|test013.png|50|2": 26.953985304939764, "BSD68|test014.png|50|2": 28.250423212803494, "BSD68|test015.png|50|2": 24.429453884443816, "BSD68|test016.png|50|2": 24.979148460049352, "BSD68|test017.png|50|2": 31.005535227074, "BSD68|test018.png|50|2": 31.095513547919182, "BSD68|test019.png|50|2": 25.79333479003846, "BSD68|test020.png|50|2": 26.03650987985152, "BSD68|test021.png|50|2": 23.171235059788096, "BSD68|test022.png|50|2": 24.408305785570136, "BSD68|test024.png|50|2": 25.53515876267189, "BSD68|test023.png|50|2": 25.854741749945585, "BSD68|test025.png|50|2": 26.54063240941873, "BSD68|test026.png|50|2": 26.002515733542243, "BSD68|test027.png|50|2": 27.396692332805472, "BSD68|test028.png|50|2": 30.65568024447513, "BSD68|test029.png|50|2": 21.168436643447198, "BSD68|test030.png|50|2": 27.229291903484352, "BSD68|test031.png|50|2": 21.558515323331285, "BSD68|test032.png|50|2": 23.69823196642111, "BSD68|test033.png|50|2": 25.15519361358008, "BSD68|test034.png|50|2": 31.458675881438516, "BSD68|test035.png|50|2": 25.693124254121177, "BSD68|test036.png|50|2": 26.504031950966823, "BSD68|test037.png|50|2": 26.513208617884438, "BSD68|test038.png|50|2": 26.47406070739485, "BSD68|test039.png|50|2": 30.87953692764598, "BSD68|test040.png|50|2": 26.92812021007354, "BSD68|test041.png|50|2": 25.442022356088195, "BSD68|test042.png|50|2": 26.98416520517928, "BSD68|test043.png|50|2": 27.355971473276973, "BSD68|test044.png|50|2": 25.660608754893538, "BSD68|test045.png|50|2": 31.992738608791434, "BSD68|test046.png|50|2": 23.418287051190365, "BSD68|test047.png|50|2": 23.8112802254232, "BSD68|test048.png|50|2": 26.852622298731745, "BSD68|test049.png|50|2": 29.70515359974624, "BSD68|test050.png|50|2": 24.40989903091475, "BSD68|test051.png|50|2": 25.875019904911667, "BSD68|test052.png|50|2": 30.28930216094603, "BSD68|test053.png|50|2": 29.44777772360794, "BSD68|test054.png|50|2": 26.489032231510347, "BSD68|test055.png|50|2": 24.75682642169574, "BSD68|test056.png|50|2": 21.24426467682104, "BSD68|test057.png|50|2": 25.94301236767192, "BSD68|test058.png|50|2": 29.3177828762786, "BSD68|test059.png|50|2": 28.651352002561037, "BSD68|test060.png|50|2": 29.149309782401133, "BSD68|test061.png|50|2": 27.96670822432627, "BSD68|test062.png|50|2": 31.75048468259775, "BSD68|test063.png|50|2": 23.518578959357725, "BSD68|test065.png|50|2": 26.363568390638545, "BSD68|test064.png|50|2": 24.385611914439064, "BSD68|test066.png|50|2": 35.51047591550612, "BSD68|test067.png|50|2": 21.57910320883095, "BSD68|test068.png|50|2": 23.053105230278295}''')   # "dataset|image|sigma|seed" -> Restormer-gray PSNR in the lift job
SEEDS = sorted({int(k.split("|")[3]) for k in REF})
print(len(REF), "reference rows; seeds", SEEDS, ";", os.cpu_count(), "CPUs")


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        for attempt in range(5):
            try:
                urllib.request.urlretrieve(url, path)
                return path
            except Exception as e:
                print(f"  retry {attempt + 1} for {url}: {e}", flush=True)
                time.sleep(5 * (attempt + 1))
        raise RuntimeError(url)
    return path


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


GH = "https://github.com/{}/releases/download/{}/{}"
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
IMAGES = {"Set12": [f"{i:02d}.png" for i in range(1, 13)], "BSD68": [f"test{i:03d}.png" for i in range(1, 69)]}
src = download("https://raw.githubusercontent.com/swz30/Restormer/68dc6ac472db26f16361150cb7a96a1bc87da93f/basicsr/models/archs/restormer_arch.py",
               f"{CACHE}/restormer_arch.py")
RestormerNet = load_module("restormer_arch", src).Restormer
TEST = {ds: [(n, np.asarray(Image.open(download(RAW.format(ds, n), f"{CACHE}/{ds}_{n}")).convert("L"))) for n in ns]
        for ds, ns in IMAGES.items()}

In [ ]:
def noisy_of(clean_u8, sigma, seed):
    """nbGray.noisy_of, called single-threaded."""
    img = np.float32(clean_u8[..., None]) / 255.
    np.random.seed(seed=seed)
    img += np.random.normal(0, sigma / 255., img.shape)
    return torch.from_numpy(img[..., 0].copy())


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    return 10 * math.log10(255 ** 2 / (a - torch.from_numpy(clean_u8).double()).pow(2).mean().item())


rows = []
t0 = time.time()
for s in SIGMAS:
    net = RestormerNet(inp_channels=1, out_channels=1, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                       ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
    w = download(GH.format("swz30/Restormer", "v1.0", f"gaussian_gray_denoising_sigma{s}.pth"), f"{CACHE}/restormer_gray_{s}.pth")
    net.load_state_dict(torch.load(w, map_location="cpu", weights_only=True)["params"], strict=True)
    net.eval().requires_grad_(False)
    for seed in SEEDS:
        for ds, items in TEST.items():
            for name, clean in items:
                y = noisy_of(clean, s, seed)
                h, wd = y.shape
                with torch.no_grad():
                    out = net(F.pad(y[None, None], (0, (-wd) % 8, 0, (-h) % 8), mode="reflect"))[0, 0, :h, :wd].clamp(0, 1)
                p = psnr(out, clean)
                ref = REF[f"{ds}|{name}|{s}|{seed}"]
                rows.append(dict(dataset=ds, image=name, sigma=s, seed=seed, psnr_cpu=p, psnr_lift_job=ref, delta=p - ref,
                                 noisy_psnr=psnr(y, clean)))
            print(f"σ {s} seed {seed} {ds}: done ({(time.time() - t0) / 60:.1f} min)", flush=True)

R = pd.DataFrame(rows)
R["same_draw"] = R.delta.abs() <= 0.003
R.to_csv(f"{OUT}/draw_check.csv", index=False)
S = R.groupby(["dataset", "sigma", "seed"]).agg(n=("delta", "size"), same=("same_draw", "sum"),
                                                max_abs_delta=("delta", lambda d: d.abs().max()))
print(S.to_string())
print("mismatching images:", R[~R.same_draw][["dataset", "image", "sigma", "seed", "delta"]].to_string(index=False))
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(all_same=bool(R.same_draw.all()), n_mismatch=int((~R.same_draw).sum()), seeds=SEEDS,
                   mismatches=R[~R.same_draw][["dataset", "image", "sigma", "seed", "delta"]].to_dict("records")), f, indent=1,
              default=float)